<a href="https://colab.research.google.com/github/Patamsetti-rajiv/create-resume-builder-basic-one/blob/main/apshe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer

# Modeling and Evaluation
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting style for clean visuals
sns.set_theme(style="whitegrid")
print("✅ Libraries successfully imported!")


In [ ]:
# Cell 2: Data Generation
np.random.seed(42)
n_samples = 1500

# Generating multi-modal features
data = {
    'income_k_usd': np.random.normal(65, 25, n_samples).clip(15, 250),       # Financial
    'sleep_hours': np.random.normal(7.2, 1.1, n_samples).clip(4, 10),        # Sleep health
    'hrv_ms': np.random.normal(55, 18, n_samples).clip(10, 120),             # Physical stress (Heart Rate Variability)
    'screen_time_hours': np.random.uniform(2, 9, n_samples),                 # Digital habits
    'social_hours': np.random.uniform(0.5, 5, n_samples),                    # Social connection
    'air_quality_aqi': np.random.uniform(12, 160, n_samples)                 # Environment
}

df = pd.DataFrame(data)

# Introduce a small percentage of missing values to mimic real-world tracking errors
mask = np.random.rand(*df.shape) < 0.05
df[mask] = np.nan

# Construct a realistic, non-linear target formula for the "Well-being Index"
# Note: Lower screen time and lower AQI values improve well-being
base_wellbeing = (
    (df['income_k_usd'].fillna(65) * 0.15) +
    (df['sleep_hours'].fillna(7.2) * 5.0) +
    (df['hrv_ms'].fillna(55) * 0.25) -
    (df['screen_time_hours'].fillna(5) * 2.0) +
    (df['social_hours'].fillna(2) * 3.5) -
    (df['air_quality_aqi'].fillna(75) * 0.08)
)

# Normalize the target cleanly between 0.0 and 100.0
df['well_being_index'] = (base_wellbeing - base_wellbeing.min()) / (base_wellbeing.max() - base_wellbeing.min()) * 100

print(f"✅ Dataset generated with shape: {df.shape}")
df.head()


In [ ]:
# Cell 3: Data Inspection
print("--- Missing Values Per Feature ---")
print(df.isnull().sum())

print("\n--- Target Variable Summary Statistics ---")
print(df['well_being_index'].describe())

# Plot target distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['well_being_index'], kde=True, color='teal')
plt.title("Distribution of the Well-being Index Target")
plt.xlabel("Well-being Index Score (0-100)")
plt.ylabel("Count")
plt.show()


In [ ]:
# Cell 4: Train/Test Split
X = df.drop(columns=['well_being_index'])
y = df['well_being_index']

# 80% training data, 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Training set size: {X_train.shape[0]} samples")
print(f"✅ Testing set size: {X_test.shape[0]} samples")


In [ ]:
# Cell 5: Preprocessing
# 1. Initialize transformers
imputer = KNNImputer(n_neighbors=5)
scaler = StandardScaler()

# 2. Fit and transform training data
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 3. Transform test data using training parameters (prevents data leakage)
X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to dataframes for easy interpretation
X_train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_final = pd.DataFrame(X_test_scaled, columns=X.columns)

print("✅ Data preprocessing complete. No missing values remain.")
X_train_final.head()


In [ ]:
# Cell 6: Training
print("Training the Gradient Boosting Regressor...")

# Initialize model with standard hyperparameters
wb_model = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=4,
    random_state=42
)

# Fit model to scaled training data
wb_model.fit(X_train_final, y_train)

print("✅ Model training successfully completed!")


In [ ]:
# Cell 7: Model Evaluation
# Generate predictions on test data
y_pred = wb_model.predict(X_test_final)

# Calculate regression performance indicators
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Test Performance Metrics ---")
print(f"Mean Absolute Error (MAE)      : {mae:.3f} points")
print(f"Root Mean Squared Error (RMSE)  : {rmse:.3f} points")
print(f"Coefficient of Determination(R²): {r2:.4f}")


In [ ]:
# Cell 8: Plotting Metrics
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Actual vs Predicted values
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, color='indigo', ax=ax[0])
ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax[0].set_title("Actual vs. Predicted Well-being Index")
ax[0].set_xlabel("Actual Index")
ax[0].set_ylabel("Predicted Index")

# Plot 2: Feature Importance
importances = wb_model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_ranking = pd.DataFrame({
    'Feature': X.columns[indices],
    'Importance': importances[indices]
})

sns.barplot(x='Importance', y='Feature', data=feature_ranking, palette='viridis', ax=ax[1])
ax[1].set_title("Feature Importance Breakdown")
ax[1].set_xlabel("Relative Gini Importance Score")

plt.tight_layout()
plt.show()


In [ ]:
# Cell 9: Single-user Prediction Sandbox
# Input raw metric values for a custom single user
single_user_raw = pd.DataFrame([{
    'income_k_usd': 85.0,
    'sleep_hours': 8.0,
    'hrv_ms': 65.0,
    'screen_time_hours': 3.5,
    'social_hours': 4.0,
    'air_quality_aqi': 42.0
}])

print("--- Raw Input User Features ---")
print(single_user_raw.to_string(index=False))

# Pass input through pipeline structures built in cell 5
user_imputed = imputer.transform(single_user_raw)
user_scaled = scaler.transform(user_imputed)

# Predict score
calculated_score = wb_model.predict(user_scaled)[0]

print("\n--- Output Summary Report ---")
print(f"Computed Comprehensive Well-being Score: {calculated_score:.1f} / 100.0")


In [ ]:
# Cell 10: Interactive UI Dashboard
import ipywidgets as widgets
from ipywidgets import Layout, Box, VBox, HBox
from IPython.display import display, clear_output

# Create an output region to display the calculated result
output_region = widgets.Output()

# Title and header layout elements
title_html = widgets.HTML(
    value="<h2 style='color:#1e3d59; font-family:Arial; margin-bottom:5px;'>📊 Well-being Index Predictor Dashboard</h2>"
          "<p style='color:#17b978; font-family:Arial; margin-top:0px;'>Adjust the lifestyle sliders below to compute your real-time well-being score.</p>"
)

# Custom stylings for consistency
slider_layout = Layout(width='450px')
style = {'description_width': '180px'}

# 1. Define interactive dashboard widgets
slider_income = widgets.FloatSlider(value=65.0, min=15.0, max=250.0, step=1.0, description='Annual Income ($k):', style=style, layout=slider_layout)
slider_sleep = widgets.FloatSlider(value=7.2, min=4.0, max=10.0, step=0.1, description='Daily Sleep (Hours):', style=style, layout=slider_layout)
slider_hrv = widgets.FloatSlider(value=55.0, min=10.0, max=120.0, step=1.0, description='Heart Rate Var. (HRV ms):', style=style, layout=slider_layout)
slider_screen = widgets.FloatSlider(value=5.0, min=2.0, max=9.0, step=0.1, description='Screen Time (Hours):', style=style, layout=slider_layout)
slider_social = widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.1, description='Social Interaction (Hours):', style=style, layout=slider_layout)
slider_aqi = widgets.FloatSlider(value=75.0, min=12.0, max=160.0, step=1.0, description='Air Quality Index (AQI):', style=style, layout=slider_layout)

predict_button = widgets.Button(
    description='Calculate Well-being Index',
    button_style='success',
    tooltip='Click to run ML inference pipeline',
    icon='check',
    layout=Layout(width='250px', height='40px', margin='20px 0px 0px 180px')
)

# 2. Calculation logic mapped to your model
def calculate_well_being(button_click_event):
    with output_region:
        clear_output() # Refresh the output block

        # Read parameters straight from the user interface
        user_metrics = pd.DataFrame([{
            'income_k_usd': slider_income.value,
            'sleep_hours': slider_sleep.value,
            'hrv_ms': slider_hrv.value,
            'screen_time_hours': slider_screen.value,
            'social_hours': slider_social.value,
            'air_quality_aqi': slider_aqi.value
        }])

        # Process inputs using the transformers engineered in Cell 5
        user_imputed = imputer.transform(user_metrics)
        user_scaled = scaler.transform(user_imputed)

        # Run ML model estimation from Cell 6
        predicted_index = wb_model.predict(user_scaled)[0]

        # Format beautiful user feedback HTML
        if predicted_index >= 70:
            color_theme, status = "#17b978", "Excellent 🌟"
        elif predicted_index >= 45:
            color_theme, status = "#ff9f43", "Moderate 😐"
        else:
            color_theme, status = "#ee5253", "Needs Attention ⚠️"

        display(widgets.HTML(
            value=f"<div style='border: 2px solid {color_theme}; padding: 15px; border-radius: 8px; max-width: 450px; background-color: #f9f9f9; margin-top:15px;'>"
                  f"<h3 style='margin: 0 0 10px 0; color: #2f3640; font-family:Arial;'>Result Summary:</h3>"
                  f"<p style='font-family:Arial; font-size: 16px; margin: 5px 0;'>Predicted Well-being Score: <b style='color:{color_theme}; font-size:20px;'>{predicted_index:.1f} / 100.0</b></p>"
                  f"<p style='font-family:Arial; font-size: 14px; margin: 5px 0;'>Status Class: <span style='font-weight:bold; color:{color_theme};'>{status}</span></p>"
                  f"</div>"
        ))

# Bind the action click trigger to our handler function
predict_button.on_click(calculate_well_being)

# 3. Stack inputs cleanly into a neat vertical dashboard form
ui_form = VBox([
    title_html,
    slider_income,
    slider_sleep,
    slider_hrv,
    slider_screen,
    slider_social,
    slider_aqi,
    predict_button,
    output_region
], layout=Layout(padding='15px', background_color='#f7f9fa', border='1px solid #e1e8ed', border_radius='10px', width='520px'))

# Run and show dashboard interface
display(ui_form)


In [ ]:
# Clear the notebook's active widget state cache to resolve the GitHub rendering bug
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True --clear-output --inplace /content/*.ipynb
